# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya: Demonstrating AI-Ready Data Standards for Data Infrastructure in Africa Exploration with `mlcroissant`
This notebook provides a step-by-step template for loading and exploring the FAIR² Mental Health Survey Dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

In [ ]:
# Ensure `mlcroissant` is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import pprint

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)

# Access metadata object
meta = dataset.metadata
print(f"{meta.name}\n\n{meta.description}")
print(f"\nPublished: {meta.datePublished}")
print(f"License: {meta.license}")

## 2. Data Overview
Review available record sets, fields, and their `@id`s.

In [ ]:
# List available record sets
print("Available record sets (by @id):")
recordsets = list(dataset.list_record_sets())
for rs in recordsets:
    print(f"- {rs['@id']} (name: {rs.get('name', 'N/A')})")

# For each record set, show fields and columns
print("\nFields and columns by record set:")
for rs in recordsets:
    print(f"\nRecord Set: {rs['@id']} ({rs.get('name','N/A')})")
    fields = dataset.list_fields(record_set=rs['@id'])
    for field in fields:
        print(f"  Field: {field['@id']} (name: {field.get('name','N/A')})");
        if 'column' in field:
            cols = field['column'] if isinstance(field['column'], list) else [field['column']]
            for col in cols:
                print(f"    Column: {col['@id']} (name: {col.get('name','N/A')})")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. All references are by `@id`.

In [ ]:
# Get all the record set @id values
record_set_ids = [rs['@id'] for rs in recordsets]
print("Record sets to load:", record_set_ids)

# Load all data into DataFrames by record set @id
dataframes = {}
for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"Loaded {len(df)} records for {record_set_id}.")

# Let's print the columns of the primary record set
main_record_set = record_set_ids[0] if record_set_ids else None
if main_record_set:
    print(f"\nColumns in main record set ({main_record_set}):")
    print(dataframes[main_record_set].columns.tolist())
    display(dataframes[main_record_set].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering based on field values, normalizing fields, and grouping by key attributes. All features referenced by their `@id` or DataFrame columns derived from them.

In [ ]:
# Choose a numeric field for EDA
# We'll programmatically select the GAD-7 total score (if present, by column name containing 'gad7' and 'sum' or 'score')
main_df = dataframes[main_record_set]
gad7_col = None
for col in main_df.columns:
    if 'gad' in col.lower() and ('sum' in col.lower() or 'score' in col.lower()):
        gad7_col = col
        break
if gad7_col is None:
    # fallback to first numeric
    gad7_col = main_df.select_dtypes('number').columns[0]
print(f"Selected numeric field for analysis: {gad7_col}")

threshold = main_df[gad7_col].mean() + main_df[gad7_col].std()  # Use a dynamic threshold (mean + 1 std)
filtered_df = main_df[main_df[gad7_col] > threshold]
print(f"Filtered records with {gad7_col} > {threshold:.2f} (total: {len(filtered_df)} records):")
display(filtered_df.head())

# Normalize the field
normalized_col = f"{gad7_col}_normalized"
filtered_df[normalized_col] = (filtered_df[gad7_col] - main_df[gad7_col].mean()) / main_df[gad7_col].std()
print(f"Normalized {gad7_col} for filtered records:")
display(filtered_df[[gad7_col, normalized_col]].head())

# Group by a demographic field if possible
group_field = None
for col in main_df.columns:
    if ('gender' in col.lower()) or ('sex' in col.lower()):
        group_field = col
        break
if group_field:
    grouped_df = filtered_df.groupby(group_field)[gad7_col].mean().reset_index()
    print(f"\nMean {gad7_col} by {group_field} (filtered records):")
    display(grouped_df.head())

## 5. Visualization
Visualize distribution of the chosen numeric field (e.g., GAD-7 sum/score) and breakdown by a key attribute.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Distribution of GAD-7
plt.figure(figsize=(8,5))
sns.histplot(main_df[gad7_col], bins=15, kde=True)
plt.title(f"Distribution of {gad7_col}")
plt.xlabel(gad7_col)
plt.ylabel("Count")
plt.show()

# If grouping field exists, show boxplot
if group_field:
    plt.figure(figsize=(8,5))
    sns.boxplot(x=main_df[group_field], y=main_df[gad7_col], showmeans=True)
    plt.title(f"{gad7_col} by {group_field}")
    plt.xlabel(group_field)
    plt.ylabel(gad7_col)
    plt.show()

## 6. Conclusion
In this notebook, we:

- Loaded the FAIR² Mental Health Survey Dataset using `mlcroissant` referencing all entities by `@id`
- Explored metadata, record sets, fields, and columns
- Loaded data into DataFrames and identified key numeric fields (e.g., GAD-7 sum score)
- Applied data filtering, normalization, and grouped analysis using demographic fields
- Visualized score distributions and differences across demographic groups where available

This demonstrates programmatic, schema-driven data exploration leveraging the Croissant standard. For further analysis, refer to additional fields and record sets by their `@id` as needed.